Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\a812616\AgenticAI\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Read all pdfs inside the directory
def process_all_pdfs(pdf_directory):
    """process all pdf files in a directory"""
    all_documents =  []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #Add source information to the metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

In [3]:
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: test.pdf
Loaded 1 pages

Processing: test1.pdf
Loaded 2 pages

 Total documents loaded: 3


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-10-14T12:30:02+05:30', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_enabled': 'true', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_setdate': '2025-10-14T06:59:33Z', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_method': 'Privileged', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_name': 'Public', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_siteid': '33440fc6-b7c7-412c-bb73-0e70b0198d5a', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_actionid': 'b4f41f9b-6133-49ee-a5de-0d875bb92001', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_contentbits': '0', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_tag': '10, 0, 1, 1', 'author': 'Davis Boban', 'moddate': '2025-10-14T12:30:02+05:30', 'source': '..\\data\\pdf\\test.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'test.pdf', 'file_type': 'pdf'}, page_content='Ag

In [5]:
#Text splitting get into chunks
"""When splitting a document into smaller chunks, chunk_overlap defines
 how many characters from the end of one chunk 
are repeated at the beginning of the next chunk."""

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n","\n"," ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {(len(documents))} documents into {len(split_docs)} chunks")

    #show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [6]:
chunks = split_documents(all_pdf_documents)
chunks

Split 3 documents into 5 chunks

Example chunk:
Content: Agentic AI refers to artificial intelligence systems that exhibit agency—meaning they 
can autonomously pursue goals, make decisions, and take actions in dynamic 
environments, often with minimal huma...
Metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-10-14T12:30:02+05:30', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_enabled': 'true', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_setdate': '2025-10-14T06:59:33Z', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_method': 'Privileged', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_name': 'Public', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_siteid': '33440fc6-b7c7-412c-bb73-0e70b0198d5a', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_actionid': 'b4f41f9b-6133-49ee-a5de-0d875bb92001', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_contentbits': '0', 'msip_label_609d143b-9e20-4325-a

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-10-14T12:30:02+05:30', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_enabled': 'true', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_setdate': '2025-10-14T06:59:33Z', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_method': 'Privileged', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_name': 'Public', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_siteid': '33440fc6-b7c7-412c-bb73-0e70b0198d5a', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_actionid': 'b4f41f9b-6133-49ee-a5de-0d875bb92001', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_contentbits': '0', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_tag': '10, 0, 1, 1', 'author': 'Davis Boban', 'moddate': '2025-10-14T12:30:02+05:30', 'source': '..\\data\\pdf\\test.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'test.pdf', 'file_type': 'pdf'}, page_content='Ag

Embedding and VectorstoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer #embedding model will be available here
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): #this model helps to convert text to vector of ~384 dimensions
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model() #load the model

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
             texts: List of text strings to embed

        Returns:
             numpy array of embeddings with shape(len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
def get_embeddings_dimension(self) -> int:
        """Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()

In [9]:
#Initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model:all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


VectorStore

In [10]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
           collection_name: Name of the ChromaDB collection
           persist_directory: Directory to persist the vector store stored in the harddisk

        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            #Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
           documents: List of LangChain documents
           embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        #Prepare data for ChromaDB
        ids =[]
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents,embeddings)):
            #Generate Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            #Document content
            documents_text.append(doc.page_content)

            #Embeddings
            embeddings_list.append(embedding.tolist())

        #Add to collection
        try:
            self.collection.add(
            ids = ids,
            embeddings = embeddings_list,
            metadatas= metadatas,
            documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
                                             

In [11]:
vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 5


In [12]:
chunks

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-10-14T12:30:02+05:30', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_enabled': 'true', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_setdate': '2025-10-14T06:59:33Z', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_method': 'Privileged', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_name': 'Public', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_siteid': '33440fc6-b7c7-412c-bb73-0e70b0198d5a', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_actionid': 'b4f41f9b-6133-49ee-a5de-0d875bb92001', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_contentbits': '0', 'msip_label_609d143b-9e20-4325-a486-915934a7f97c_tag': '10, 0, 1, 1', 'author': 'Davis Boban', 'moddate': '2025-10-14T12:30:02+05:30', 'source': '..\\data\\pdf\\test.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'test.pdf', 'file_type': 'pdf'}, page_content='Ag

In [13]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks] #generate list of chunks
texts


['Agentic AI refers to artificial intelligence systems that exhibit agency—meaning they \ncan autonomously pursue goals, make decisions, and take actions in dynamic \nenvironments, often with minimal human intervention. These systems go beyond \nreactive behavior and are designed to be proactive, adaptive, and goal-directed. \nKey Characteristics of Agentic AI: \n1. Autonomy: Operates independently, making decisions without constant human \ninput. \n2. Goal-Oriented Behavior: Pursues specific objectives, often defined by users or \nlearned through experience. \n3. Planning and Reasoning: Can plan steps to achieve goals, reason about \nconsequences, and adjust strategies. \n4. Context Awareness: Understands and adapts to changing environments or user \nneeds. \n5. Learning and Improvement: Learns from interactions and outcomes to improve \nfuture performance. \nExamples of Agentic AI: \n• Personal AI assistants that manage tasks, schedule meetings, and make \ndecisions based on user pre

In [14]:
#Generate embedding
embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 5 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.62it/s]

Generated embeddings with shape: (5, 384)


In [15]:
#store in the vector db
vectorstore.add_documents(chunks, embeddings)

Adding 5 documents to vector store...
Successfully added 5 documents to vector store
Total documents in collection: 10


Retriever Pipeline from VectorStore

In [23]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query, and helps getting the context

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, score threshold: {score_threshold}")

        #Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings= [query_embedding.tolist()],
                n_results=top_k
            )

            #process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #Convert distance to similarity score(ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id" : doc_id,
                            'content' : document,
                            'similarity_score' : similarity_score,
                            'distance' : distance,
                            'rank': i+1
                        })
                    
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                
            else:
                print("No documents found")

            return retrieved_docs #context is retrieved in the form of list
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
                

In [24]:
rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever

In [25]:
rag_retriever.retrieve(query="What is RAG") #retrieve context

Retrieving documents for query: 'What is RAG'
Top K: 5, score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 91.56it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_a7e0c5a3_3',
  'content': 'outdated or limited. RAG enhances these models by: \n• Providing up-to-date information \n• Reducing hallucinations (i.e., making up facts) \n• Improving factual accuracy \n• Allowing domain-specific customization (e.g., using company documents or \nscientific papers) \n \nHow RAG Works (Simplified Flow): \n1. User Query →  \n2. Retriever searches a document store (e.g., vector database like FAISS or \nElasticsearch) →  \n3. Top-k relevant documents are selected →  \n4. Generator (e.g., a transformer model) uses these documents to craft a \nresponse. \n \n Use Cases of RAG:',
  'similarity_score': 0.2890356779098511,
  'distance': 0.7109643220901489,
  'rank': 1},
 {'id': 'doc_df04ce8c_3',
  'content': 'outdated or limited. RAG enhances these models by: \n• Providing up-to-date information \n• Reducing hallucinations (i.e., making up facts) \n• Improving factual accuracy \n• Allowing domain-specific customization (e.g., using company documents or

Integration of VectorDB context pipeline with LLM output

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
#Initialize the Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
llm = ChatGroq(groq_api_key = groq_api_key, model_name = "gemma2-9b-it", temperature = 0.1, max_tokens=1024)

In [ ]:
#RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k =3):
    #retrieve context
    results = retriever.retrieve(query, top_k = top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to the answer question."
    
    #generate the answer using GROQ LLM
    prompt = f"""Use the following context to answer the question precisely
    Context: {context}

    Question: {query}
    
    Answer: """

    response = llm.invoke([prompt.format(context = context, query = query)])
    return response.content

In [ ]:
answer = rag_simple("What is attention mechanism?", rag_retriever, llm)
print(answer)